# Free Colab eval (NO Unsloth) — SFT model at 2500 tokens

Runs on a free **T4 GPU** with plain transformers + peft (no Unsloth/torchao).

**Setup:** Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom.
You'll upload `sft-adapter.zip` and `eval.jsonl` in Cell 2.

## 1. Install (Colab-stable, no Unsloth)

In [1]:
!pip install -q -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


## 2. Upload sft-adapter.zip + eval.jsonl, then unzip

In [3]:
import os, zipfile
# files already uploaded to /content — just unzip the adapter
if os.path.exists("sft-adapter.zip") and not os.path.exists("sft-adapter"):
    with zipfile.ZipFile("sft-adapter.zip") as z:
        z.extractall("sft-adapter")
print("adapter files:", os.listdir("sft-adapter") if os.path.exists("sft-adapter") else "MISSING")
print("eval.jsonl present:", os.path.exists("eval.jsonl"))


adapter files: ['sft-adapter']
eval.jsonl present: True


## 3. Load base Qwen2.5-7B (4-bit) + your adapter

## 4. Evaluate (2500 tokens) + OVERALL ACCURACY

In [4]:
import torch, warnings, logging
warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
BASE = "Qwen/Qwen2.5-7B-Instruct"
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, "sft-adapter")
model.eval()
tok = AutoTokenizer.from_pretrained("sft-adapter")
if tok.pad_token_id is None: tok.pad_token = tok.eos_token
print("model + adapter loaded on", model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

ValueError: Can't find 'adapter_config.json' at 'sft-adapter'

In [5]:
import os, zipfile, shutil
if os.path.exists("sft-adapter"):
    shutil.rmtree("sft-adapter")          # clean any partial/empty folder
with zipfile.ZipFile("sft-adapter.zip") as z:
    z.extractall("sft-adapter")

ADAPTER_PATH = None
for root, dirs, fs in os.walk("sft-adapter"):
    if "adapter_config.json" in fs:
        ADAPTER_PATH = root
        break
print("ADAPTER_PATH =", ADAPTER_PATH)
print("contents:", os.listdir(ADAPTER_PATH) if ADAPTER_PATH else "adapter_config.json NOT FOUND")


ADAPTER_PATH = sft-adapter/sft-adapter
contents: ['chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'README.md', 'tokenizer.json']


In [6]:
from peft import PeftModel
from transformers import AutoTokenizer

model = PeftModel.from_pretrained(base, ADAPTER_PATH)   # 'base' is still in memory
model.eval()
tok = AutoTokenizer.from_pretrained(ADAPTER_PATH)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
print("✅ adapter loaded from", ADAPTER_PATH, "on", model.device)


✅ adapter loaded from sft-adapter/sft-adapter on cuda:0


In [10]:
import json, torch

SYSTEM = ("You are an expert fitness coach. Given a user profile, output ONLY a JSON "
    "object matching this schema: goal, experience (beginner|intermediate|advanced), "
    "daily_schedule{wake, workout{time,type}, meals[{time,name,focus}], sleep{target,hours}}, "
    "weekly_workouts[{day, focus, exercises[{name,sets,reps,rest_seconds,demo_image,why}]}], "
    "nutrition{daily_macros{calories,protein_g,carbs_g,fat_g}, "
    "example_day[{food,grams,calories,protein_g}], grocery_list[]}, disclaimer. "
    "Use realistic exercises and macros. Set demo_image to null (filled later). "
    "For beginners, fill 'why' with a short reason. Always include a disclaimer that "
    "this is general guidance, not medical advice. If the user reports an injury, "
    "avoid contraindicated movements.")

CONTRA = {"knee pain":["squat","lunge","leg press"],
          "shoulder impingement":["overhead press","upright row","bench press"],
          "lower back pain":["deadlift","good morning","bent over row"]}

def extract_json(t):
    try: return json.loads(t)
    except Exception: pass
    i, j = t.find("{"), t.rfind("}")
    if i >= 0 and j > i:
        try: return json.loads(t[i:j+1])
        except Exception: return None
    return None

def exercise_names(plan):
    names = []
    wk = plan.get("weekly_workouts")
    if isinstance(wk, list):
        for d in wk:
            if isinstance(d, dict):
                for e in d.get("exercises", []):
                    if isinstance(e, dict): names.append(str(e.get("name","")))
                    elif isinstance(e, str): names.append(e)
    for v in plan.values():
        if isinstance(v, dict) and isinstance(v.get("workouts"), list):
            for e in v["workouts"]:
                if isinstance(e, dict): names.append(str(e.get("exercise") or e.get("name","")))
    return [n.lower() for n in names if n]

def schema_ok(plan):
    wk = plan.get("weekly_workouts")
    if not (isinstance(wk, list) and wk and isinstance(wk[0], dict)): return False
    exs = wk[0].get("exercises")
    return isinstance(exs, list) and all(isinstance(e, dict) and "name" in e for e in exs) and "nutrition" in plan

def avoids_injury(names, injury):
    if not injury or injury == "None": return True
    return not any(b in n for n in names for b in CONTRA.get(injury, []))

def respects_equipment(names, eq):
    if eq != "bodyweight only": return True
    banned = ["barbell","machine","cable","leg press"]
    return not any(b in n for n in names for b in banned)

def gen(p):
    u = (f"Profile: {p['age']}yo, {p['weight_kg']}kg, goal: {p['goal']}, "
         f"equipment: {p['equipment']}, diet: {p['diet']}, experience: {p['experience']}, "
         f"injury: {p['injury']}, {p['days_per_week']} days/week. Return the JSON plan.")
    msgs = [{"role":"system","content":SYSTEM}, {"role":"user","content":u}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    in_len = enc["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=2500, do_sample=True,
                             temperature=0.7, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][in_len:], skip_special_tokens=True)

rows = [json.loads(l) for l in open("eval.jsonl")][:40]
valid = schema = eq = inj = passed = 0
for k, r in enumerate(rows, 1):
    p = r["profile"]; plan = extract_json(gen(p))
    if plan:
        valid += 1
        s = schema_ok(plan); names = exercise_names(plan)
        e = respects_equipment(names, p["equipment"]); i2 = avoids_injury(names, p["injury"])
        schema += int(s); eq += int(e); inj += int(i2)
        if s and e and i2: passed += 1
    if k % 10 == 0: print(f"...{k}/{len(rows)}")
n = len(rows)
print(f"\nSFT (no-Unsloth, 2500 tokens) on {n} held-out profiles:")
print(f"  valid JSON:  {valid}/{n} = {valid/n:.0%}")
print(f"  schema:      {schema}/{n} = {schema/n:.0%}")
print(f"  equipment:   {eq}/{n} = {eq/n:.0%}")
print(f"  injury safe: {inj}/{n} = {inj/n:.0%}")
print(f"\n  >>> OVERALL ACCURACY (all pass): {passed}/{n} = {passed/n:.0%} <<<")


...10/40
...20/40
...30/40


KeyboardInterrupt: 

In [11]:
d = k  # however many finished
print(f"On {d} profiles:")
print(f"  valid JSON:  {valid/d:.0%}")
print(f"  schema:      {schema/d:.0%}")
print(f"  equipment:   {eq/d:.0%}")
print(f"  injury safe: {inj/d:.0%}")
print(f"  >>> OVERALL ACCURACY: {passed}/{d} = {passed/d:.0%} <<<")


On 37 profiles:
  valid JSON:  97%
  schema:      97%
  equipment:   97%
  injury safe: 84%
  >>> OVERALL ACCURACY: 31/37 = 84% <<<
